In [2]:
from pathlib import Path
import sys
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
repo_root

WindowsPath('C:/Users/leungk/OneDrive - EllisDon Corporation/Documents/Other/github_repos/CBG_analysis')

In [32]:
# %pip install pyyaml requests pyarrow==15.0.2 fastparquet --force-reinstall

# Run Full Pipeline
Execute the deterministic pipeline end-to-end and surface summary metrics so meals can be mapped to glucose outcomes with audit artifacts.

In [3]:
from pathlib import Path
import importlib
from src import io_excel, run_pipeline, defaults, usda_client, units, validate, aggregate

# Reload to pick up latest code changes
importlib.reload(units)
importlib.reload(aggregate)
importlib.reload(io_excel)
importlib.reload(defaults)
importlib.reload(usda_client)
importlib.reload(validate)
importlib.reload(run_pipeline)
 

<module 'src.run_pipeline' from 'C:\\Users\\leungk\\OneDrive - EllisDon Corporation\\Documents\\Other\\github_repos\\CBG_analysis\\src\\run_pipeline.py'>

In [ ]:
excel_path = repo_root / 'data' / 'source_data' / '20251218_Trudy_Meals.xlsx'
api_keys_path = repo_root / 'config' / 'api_keys.json'
defaults_path = repo_root / 'config' / 'defaults_food_items.yaml'
grams_overrides_path = repo_root / 'config' / 'grams_overrides.yaml'
output_dir = repo_root / 'data' / 'outputs'

print('Running pipeline end-to-end...')
print('Inputs -> excel:', excel_path, '| defaults:', defaults_path, '| grams overrides:', grams_overrides_path, '| outputs:', output_dir)
summary = run_pipeline.run_pipeline(
    excel_path=excel_path,
    api_keys_path=api_keys_path,
    defaults_path=defaults_path,
    grams_overrides_path=grams_overrides_path,
    output_dir=output_dir,
    debug=False,
    throttle_enabled=True,
    throttle_max_seconds=None,
    throttle_batch_size = 10,
    throttle_batch_pause_seconds = 5,
)
print('Pipeline complete. Summary:')
print(summary)
print(f'Outputs written to {output_dir}')

[INFO] 2026-02-07 08:16:06,679 - Pipeline start


[INFO] 2026-02-07 08:16:06,683 - Loading Excel from C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\data\source_data\20251218_Trudy_Meals.xlsx
C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\src\io_excel.py:87: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time_parsed = pd.to_datetime(time_series, errors="coerce").dt.time
[INFO] 2026-02-07 08:16:06,728 - Loaded clean events: 206 rows
[INFO] 2026-02-07 08:16:06,735 - Exploding meals into item rows


Running pipeline end-to-end...
Inputs -> excel: C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\data\source_data\20251218_Trudy_Meals.xlsx | defaults: C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\config\defaults_food_items.yaml | grams overrides: C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\config\grams_overrides.yaml | outputs: C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\data\outputs


[INFO] 2026-02-07 08:16:06,889 - Items exploded: 759 rows across 205 meals
C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\src\validate.py:182: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  residual_mask = items.get("food_name_std", "").astype(str).str.contains(pattern, regex=True, na=False)
[INFO] 2026-02-07 08:16:06,889 - Applying default rules from C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\config\defaults_food_items.yaml
[INFO] 2026-02-07 08:16:06,936 - Defaults applied; assumed_100g_flag rate=nan
[INFO] 2026-02-07 08:16:06,937 - Computing grams for items
[INFO] 2026-02-07 08:16:06,955 - Enriching with USDA cache/API using C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\data\cache_data\parquet\food_nutrition_cache.parquet
[INFO] 2026-02-07 08:16:06,955 - Loading USD

Pipeline complete. Summary:
{'clean_events_rows': 206, 'items_rows': 759, 'meal_rows': 205, 'model_rows': 205, 'assumptions_rows': 284, 'manual_review_rows': 671}
Outputs written to C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\data\outputs


In [34]:
# Quick health checks
import pandas as pd
assumptions = pd.read_csv(output_dir / 'assumptions_report.csv')
manual = pd.read_csv(output_dir / 'manual_review_foods.csv')
print('Assumptions rows:', len(assumptions))
print('Manual review rows:', len(manual))
print('Top assumption reasons:')
print(assumptions['assumption_reason'].value_counts().head(10))
print('Manual review sample:')
print(manual.head(10))

Assumptions rows: 284
Manual review rows: 671
Top assumption reasons:
assumption_reason
inferred_unit_count_missing_unit    108
default_missing_qty_unit_100g       102
inferred_unit_cup_missing_unit       53
missing_grams_per_count              15
default_flax_hemp_3tsp                4
default_chia_3tsp                     2
Name: count, dtype: int64
Manual review sample:
  food_text_raw food_name_std  meal_id      event_id  \
0    5 pretzels      pretzels       21  d68a8c8b6c55   
1    5 pretzels      pretzels       22  fe8cc9a59944   
2    3 pretzels      pretzels       27  ae1d94a1e50d   
3    3 pretzels      pretzels       31  66c9f9812c83   
4    5 pretzels      pretzels       34  eb483700af58   
5    2 pretzels      pretzels       40  48e233f74ba8   
6    3 pretzels      pretzels       49  131dc62edd22   
7    2 pretzels      pretzels       60  f563ab9f2332   
8    2 pretzels      pretzels       85  3d639e953fb1   
9    3 pretzels      pretzels       91  25bced6a10c1   

       

In [35]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [36]:
# manual.iloc[20:100]
manual.head(10)

,food_text_raw,food_name_std,meal_id,event_id,assumption_reason,usda_match_status
0,5 pretzels,pretzels,21,d68a8c8b6c55,inferred_unit_count_missing_unit,matched
1,5 pretzels,pretzels,22,fe8cc9a59944,inferred_unit_count_missing_unit,matched
2,3 pretzels,pretzels,27,ae1d94a1e50d,inferred_unit_count_missing_unit,matched
3,3 pretzels,pretzels,31,66c9f9812c83,inferred_unit_count_missing_unit,matched
4,5 pretzels,pretzels,34,eb483700af58,inferred_unit_count_missing_unit,matched
5,2 pretzels,pretzels,40,48e233f74ba8,inferred_unit_count_missing_unit,matched
6,3 pretzels,pretzels,49,131dc62edd22,inferred_unit_count_missing_unit,matched
7,2 pretzels,pretzels,60,f563ab9f2332,inferred_unit_count_missing_unit,matched
8,2 pretzels,pretzels,85,3d639e953fb1,inferred_unit_count_missing_unit,matched
9,3 pretzels,pretzels,91,25bced6a10c1,inferred_unit_count_missing_unit,matched


In [5]:
import requests
import json
from pathlib import Path

CONFIG_PATH = Path("../config/api_keys.json")
with open(CONFIG_PATH, "r") as f:
    API_KEYS = json.load(f)

USDA_API_KEY = API_KEYS.get("USDA_food_data")

def usda_search(food_query: str, page_size: int = 5):
    """Search USDA FoodData Central for a food query and return top matches with macro fields."""
    if not USDA_API_KEY:
        raise ValueError("USDA_food_data key missing in config/api_keys.json")
    url = "https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {
        "api_key": USDA_API_KEY,
        "query": food_query,
        "pageSize": page_size,
    }
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    foods = data.get("foods", [])
    results = []
    for food in foods:
        nutrients = {}
        for nut in food.get("foodNutrients", []):
            name = nut.get("nutrientName")
            value = nut.get("value")
            unit = nut.get("unitName")
            nutrients[name] = f"{value} {unit}"
        results.append({
            "fdcId": food.get("fdcId"),
            "description": food.get("description"),
            "dataType": food.get("dataType"),
            "nutrients": nutrients,
        })
    return results

In [6]:
# Helper: search USDA for a manual-review food (fallback to 'blueberry') and print a compact table with similarity to query
import difflib
query_food_item= "barbeque pork"
# try:
#     if not manual.empty and "food_name_std" in manual.columns:
#         first_match = manual["food_name_std"].sample(1).dropna()
#         if not first_match.empty:
#             query_food_item = first_match.iloc[0]
# except Exception:
#     pass

results = usda_search(query_food_item, page_size=5)
print(f"Query: {query_food_item}")

def _grams_from_nutrient(value):
    """Extract numeric grams from strings like '12 G' or '12 g'."""
    if value is None:
        return None
    try:
        return float(str(value).split()[0])
    except Exception:
        return None

if not results:
    print("No matches returned.")
else:
    rows = []
    for r in results:
        nutrients = r.get("nutrients", {})
        desc = r.get("description", "") or ""
        similarity_pct = round(difflib.SequenceMatcher(None, query_food_item.lower(), desc.lower()).ratio() * 100, 1)
        protein_g = _grams_from_nutrient(nutrients.get("Protein"))
        carbs_g = _grams_from_nutrient(nutrients.get("Carbohydrate, by difference"))
        fat_g = _grams_from_nutrient(nutrients.get("Total lipid (fat)"))
        total_macro_grams = sum(v for v in [protein_g, carbs_g, fat_g] if v is not None) if any(v is not None for v in [protein_g, carbs_g, fat_g]) else None
        rows.append({
            "fdcId": r.get("fdcId"),
            "description": desc,
            "dataType": r.get("dataType"),
            "similarity_pct": similarity_pct,
            "Energy": nutrients.get("Energy"),
            "Protein": nutrients.get("Protein"),
            "Carbs": nutrients.get("Carbohydrate, by difference"),
            "Fat": nutrients.get("Total lipid (fat)"),
            "total_macro_grams": total_macro_grams,
        })
    import pandas as pd
    display(pd.DataFrame(rows))

Query: barbeque pork


,fdcId,description,dataType,similarity_pct,Energy,Protein,Carbs,Fat,total_macro_grams
0,169157,"Pork, pickled pork hocks",SR Legacy,37.8,171 KCAL,19.1 G,0.0 G,10.5 G,29.60
1,2545213,"BARBEQUE FLAVORED PORK RINDS, BARBEQUE",Branded,51.0,500 KCAL,57.1 G,7.14 G,32.1 G,96.34
2,168287,"Pork, cured, salt pork, raw",SR Legacy,30.0,748 KCAL,5.05 G,0.0 G,80.5 G,85.55
3,2705895,"Pork, cracklings",Survey (FNDDS),27.6,569 KCAL,45.03 G,0 G,41.74 G,86.77
4,2705901,"Pork, belly",Survey (FNDDS),33.3,404 KCAL,26.58 G,0 G,32.24 G,58.82


# Phase 999